# Pneumonia Classification - Pediatric Chest X-Ray Dataset
## Training a Binary Classifier (Transfer Learning)

This notebook loads the `chest_xray` dataset (which is already organized into `train`, `test`, and `val` folders with `NORMAL` and `PNEUMONIA` subdirectories) and trains an AI to classify them.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
pip install lime

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras import layers, models
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

# =========================
# 🖥️ GPU Check & Config
# =========================
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    print(f"✅ GPU is active! Training on: {physical_devices[0]}")
    try:
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
    except RuntimeError as e:
        print(e)
else:
    print("❌ GPU not found! In Colab, go to Runtime > Change runtime type > T4 GPU.")

# =========================
# 🔒 Reproducibility
# =========================
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# =========================
# 📁 Path
# =========================
DATA_DIR = "/content/drive/MyDrive/my_4k_dataset"

# =========================
# ⚙️ Parameters
# =========================
BATCH_SIZE = 16
IMG_SIZE = (512, 512)

# =========================
# 📦 Load + Split dataset
# =========================
train_dataset = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_dataset = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_batches = tf.data.experimental.cardinality(val_dataset)
test_dataset = val_dataset.take(val_batches // 2)
val_dataset = val_dataset.skip(val_batches // 2)

class_names = train_dataset.class_names
print(f"\nClasses: {class_names}")

# =========================
# 🔄 Normalization & Augmentation
# =========================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
])

def prepare_train_images(x, y):
    # 1. Apply DenseNet preprocessing
    x = preprocess_input(x)
    # 2. Apply augmentation explicitly in training mode
    x = data_augmentation(x, training=True)
    return x, y

def prepare_val_test_images(x, y):
    # Only apply preprocessing, NO augmentation
    return preprocess_input(x), y

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.map(prepare_train_images, num_parallel_calls=AUTOTUNE).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.map(prepare_val_test_images, num_parallel_calls=AUTOTUNE).prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.map(prepare_val_test_images, num_parallel_calls=AUTOTUNE).prefetch(buffer_size=AUTOTUNE)

# =========================
# 🧠 Model (DenseNet121)
# =========================
base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(512, 512, 3)
)

base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs=base_model.input, outputs=output)

# =========================
# 🛑 Callbacks
# =========================
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_accuracy',
        save_best_only=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.3,
        patience=2,
        min_lr=1e-6
    )
]

# =========================
# 🚀 Phase 1: Train Head Only
# =========================
print("\n=== Phase 1: Training Head (Base Frozen) ===")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3), # Higher LR for training the fresh head
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history_phase1 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=callbacks
)

# =========================
# 🚀 Phase 2: Fine-Tuning
# =========================
print("\n=== Phase 2: Fine-Tuning Top 30 Layers ===")
base_model.trainable = True

# Freeze all layers EXCEPT the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with a much lower learning rate for fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history_phase2 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=callbacks
)

# =========================
# 📊 Evaluation & Metrics
# =========================
print("\n=== Final Evaluation ===")
test_loss, test_acc, test_auc = model.evaluate(test_dataset)

print(f"\nTest Accuracy: {test_acc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

# Generate Predictions for Confusion Matrix
y_true, y_pred = [], []
for images, labels in test_dataset:
    preds = model.predict(images, verbose=0)
    preds = (preds > 0.5).astype(int)
    y_true.extend(labels.numpy())
    y_pred.extend(preds.flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Confusion Matrix Plot
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - DenseNet121")
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm

print("\n=== 🛠️ Building Grad-CAM Model ===")

# 1. Target the correct layer directly from the main model
# 'relu' is the final convolutional activation layer before pooling in DenseNet121
target_layer_name = 'relu'

try:
    target_layer = model.get_layer(target_layer_name)
    # ✅ FIX: Using .output.shape instead of .output_shape
    print(f"✅ Found target layer: {target_layer.name} (Shape: {target_layer.output.shape})")
except ValueError:
    print(f"❌ Could not find layer '{target_layer_name}'. Check model.summary() for exact names.")

# 2. Create the Grad-CAM model
grad_model = tf.keras.models.Model(
    inputs=model.inputs,
    outputs=[target_layer.output, model.output]
)

# 3. Grad-CAM Generation Function
def make_gradcam_heatmap(img_array, grad_model):
    with tf.GradientTape() as tape:
        # Cast to float32 to ensure compatibility with GradientTape
        img_array = tf.cast(img_array, tf.float32)

        # Watch the image array
        tape.watch(img_array)

        # Forward pass through the Grad-CAM model
        last_conv_layer_output, preds = grad_model(img_array)

        # Since we have 1 output node (sigmoid), we target index 0
        class_channel = preds[:, 0]

    # Calculate gradients of the output neuron wrt the output feature map
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # Average the gradients spatially
    # grads shape is (1, H, W, Channels), so axis (0, 1, 2) leaves just Channels
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # Multiply each channel in the feature map by the pooled gradients
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Apply ReLU (discard negative values) and normalize between 0 and 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    # Safely handle any potential NaNs (if gradients completely zero out)
    heatmap = tf.where(tf.math.is_nan(heatmap), tf.zeros_like(heatmap), heatmap)

    return heatmap.numpy()

# ==========================================
# 🖼️ Display the Results on a Test Image
# ==========================================
print("\n=== 🔍 Generating Visualization ===")

# Get one batch of test images
for images, labels in test_dataset.take(1):
    sample_image = images[0:1] # Keep batch dimension: shape (1, 512, 512, 3)
    sample_label = int(labels[0].numpy()[0])
    break

# Generate Heatmap
heatmap = make_gradcam_heatmap(sample_image, grad_model)

# Make a prediction to display
prediction = model.predict(sample_image, verbose=0)[0][0]
pred_class = 1 if prediction > 0.5 else 0

# Visual Overlay setup
img_np = sample_image[0].numpy()

# Reverse the DenseNet preprocessing roughly to [0, 1] for visualization
# Adding 1e-5 to prevent division by zero
img_vis = (img_np - np.min(img_np)) / (np.max(img_np) - np.min(img_np) + 1e-5)

# Resize heatmap to match the original image dimensions dynamically
heatmap_resized = tf.image.resize(heatmap[..., tf.newaxis], (img_vis.shape[0], img_vis.shape[1])).numpy()
heatmap_resized = np.squeeze(heatmap_resized)

# Colorize the heatmap
cmap = cm.get_cmap("jet")
heatmap_colored = cmap(heatmap_resized)[..., :3]

# Superimpose heatmap onto original image (0.6 image, 0.4 heatmap opacity)
overlay = img_vis * 0.6 + heatmap_colored * 0.4
overlay = np.clip(overlay, 0, 1)

# Plotting
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_vis)
axes[0].set_title(f"Original Image\nActual: {class_names[sample_label]}")
axes[0].axis("off")

axes[1].imshow(heatmap_resized, cmap="jet")
axes[1].set_title("Grad-CAM Heatmap")
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title(f"Overlay\nPredicted: {class_names[pred_class]} (Conf: {prediction:.2f})")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from skimage.segmentation import mark_boundaries
import lime
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
from tensorflow.keras.applications.densenet import preprocess_input

print("\n=== 🟢 1. Loading RAW Image for LIME ===")

# We MUST load a raw image so Matplotlib and LIME can see it properly.
raw_test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=16,
    label_mode='binary'
)

# Skip to the test portion to ensure we evaluate unseen data
val_batches = tf.data.experimental.cardinality(raw_test_dataset)
raw_test_dataset = raw_test_dataset.skip(val_batches // 2)

image_found = False
for raw_images, raw_labels in raw_test_dataset.take(1):
    # Scale to standard [0, 1] range so it displays perfectly
    sample_image = raw_images[0].numpy() / 255.0
    lbl = raw_labels[0].numpy()
    sample_label = int(lbl.flat[0] if hasattr(lbl, 'flat') else lbl)
    image_found = True
    break

if not image_found:
    raise ValueError("❌ test_dataset is empty! Check your dataset splitting steps.")

print(f"✅ Image extracted successfully. True Label: {class_names[sample_label]}")

print("=== 🟢 2. Initializing Explainer ===")
explainer = lime_image.LimeImageExplainer()
segmenter = SegmentationAlgorithm('slic', n_segments=100, compactness=10, sigma=1)

def predict_function(images_array):
    # LIME passes arrays in [0, 1]. We scale up to 255, then let DenseNet preprocess it.
    images_255 = images_array * 255.0
    processed_images = preprocess_input(images_255)

    # Predict the perturbed batch
    preds = model.predict(processed_images, batch_size=16, verbose=0)
    return np.hstack((1 - preds, preds))

print(f"=== 🟢 3. Generating LIME perturbations... ===")
# Set to 300 for a quick check. Increase to 1000 for your final DS357 report.
explanation = explainer.explain_instance(
    sample_image.astype('double'),
    predict_function,
    top_labels=1,
    hide_color=0,
    num_samples=300,
    segmentation_fn=segmenter
)

print("=== 🟢 4. Extracting Superpixels ===")
predicted_class = explanation.top_labels[0]

temp_pos, mask_pos = explanation.get_image_and_mask(
    predicted_class, positive_only=True, num_features=5, hide_rest=False
)

temp_all, mask_all = explanation.get_image_and_mask(
    predicted_class, positive_only=False, num_features=5, hide_rest=False
)

print("=== 🟢 5. Plotting Results ===")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

# Matplotlib displays [0, 1] perfectly.
safe_temp_pos = np.clip(temp_pos, 0.0, 1.0)
safe_temp_all = np.clip(temp_all, 0.0, 1.0)

ax1.imshow(mark_boundaries(safe_temp_pos, mask_pos))
ax1.set_title(f"LIME: Supportive Regions Only\nPredicted: {class_names[predicted_class]} | Actual: {class_names[sample_label]}")
ax1.axis('off')

# Predict the single image for confidence score
confidence = predict_function(np.expand_dims(sample_image, axis=0))[0][predicted_class]

ax2.imshow(mark_boundaries(safe_temp_all, mask_all))
ax2.set_title(f"LIME: Top 5 Features (Green=Support, Red=Contradict)\nConfidence: {confidence:.2f}")
ax2.axis('off')

plt.tight_layout()
plt.show()
print("=== ✅ Done! ===")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications.densenet import preprocess_input

# =========================
# 📁 Paths & Parameters
# =========================
DATA_DIR = r"/content/drive/MyDrive/my_4k_dataset"
MODEL_PATH = 'best_model.keras'
IMG_SIZE = (512, 512) # 🔧 FIX 1: Must match the saved model's actual training shape
BATCH_SIZE = 16       # 🔧 Lowered to 16 to prevent OOM errors with 512x512
SEED = 42

# =========================
# 🧠 Load Pre-trained Model
# =========================
print("Loading saved model...")
model = tf.keras.models.load_model(MODEL_PATH)

# =========================
# 📦 Load Test Data
# =========================
val_dataset = image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_batches = tf.data.experimental.cardinality(val_dataset)
test_dataset = val_dataset.skip(val_batches // 2)

# 🔧 FIX 2: Removed the 1./255 rescaling map here.
# We want to pull RAW [0, 255] images so the RISE masks apply true black (0.0).

# =========================
# 🕵️‍♂️ Explainability: RISE
# =========================
def explain_with_rise(model, image_raw, num_masks=1000, grid_size=16, p_keep=0.5):
    img_h, img_w = image_raw.shape[:2]

    # 1. Generate small random grids
    small_masks = np.random.rand(num_masks, grid_size, grid_size) < p_keep
    small_masks = small_masks.astype(np.float32)

    # 2. Upsample using Bilinear Interpolation
    masks = tf.image.resize(small_masks[..., tf.newaxis], (img_h, img_w), method='bilinear')
    masks = tf.squeeze(masks).numpy()

    # 3. Apply masks to the RAW image
    masked_images = image_raw * masks[..., tf.newaxis]

    # 4. Predict on all masked images
    preds = []
    batch_size = 16 # 🔧 Keep small to avoid GPU memory crash
    for i in range(0, num_masks, batch_size):
        batch = masked_images[i : i + batch_size]

        # 🔧 FIX 3: Apply DenseNet preprocessing right before predicting
        batch_preprocessed = preprocess_input(batch)

        batch_preds = model.predict(batch_preprocessed, verbose=0)
        preds.extend(batch_preds)

    preds = np.array(preds).flatten()

    # 5. Generate the heatmap
    heatmap = np.zeros((img_h, img_w))
    for i in range(num_masks):
        heatmap += masks[i] * preds[i]

    # 6. Normalize the heatmap
    heatmap = heatmap / (num_masks * p_keep)
    heatmap -= heatmap.min()

    max_val = heatmap.max()
    if max_val == 0:
        max_val = 1e-8
    heatmap /= max_val

    return heatmap

# =========================
# 🖼️ Run and Visualize
# =========================
for images, labels in test_dataset.take(1):
    # images are currently raw [0, 255] floats
    sample_image = images[0].numpy()
    sample_label = labels[0].numpy()
    break

print(f"\nGenerating RISE heatmap for True Label: {sample_label[0]}...")
# 🔧 Note: num_masks=1000 at 512x512 will take a moment to run.
heatmap = explain_with_rise(model, sample_image, num_masks=1000, grid_size=16, p_keep=0.5)

# Plotting
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.title(f"Original Image (True Label: {sample_label[0]})")
# Divide by 255 just for pyplot to display it properly
plt.imshow(sample_image / 255.0)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("RISE Saliency Map Overlay")
plt.imshow(sample_image / 255.0)
plt.imshow(heatmap, cmap='jet', alpha=0.5)
plt.colorbar(fraction=0.046, pad=0.04)
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.densenet import preprocess_input

print("\n=== 🛠️ Building Grad-CAM Model ===")

# 1. Target the correct layer directly from the main model
target_layer_name = 'relu'

try:
    target_layer = model.get_layer(target_layer_name)
    print(f"✅ Found target layer: {target_layer.name} (Shape: {target_layer.output.shape})")
except ValueError:
    print(f"❌ Could not find layer '{target_layer_name}'. Check model.summary() for exact names.")

# 2. Create the Grad-CAM model
grad_model = tf.keras.models.Model(
    inputs=model.inputs,
    outputs=[target_layer.output, model.output]
)

# 3. Grad-CAM Generation Function
def make_gradcam_heatmap(img_array, grad_model):
    with tf.GradientTape() as tape:
        img_array = tf.cast(img_array, tf.float32)
        tape.watch(img_array)
        last_conv_layer_output, preds = grad_model(img_array)
        class_channel = preds[:, 0]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    heatmap = tf.where(tf.math.is_nan(heatmap), tf.zeros_like(heatmap), heatmap)

    return heatmap.numpy()

# ==========================================
# 🖼️ Load Custom Image & Generate Heatmap
# ==========================================
print("\n=== 🔍 Generating Visualization ===")

# 👇 CHANGE THIS TO YOUR SPECIFIC IMAGE PATH
custom_image_path = "/content/drive/MyDrive/my_4k_dataset/Pneumonia/CP_1113_3331_0054.png"

# Optional: Set what the actual label is manually so the plot title is correct
actual_label = "Pneumonia"

# Load and format the image to 512x512
img = image.load_img(custom_image_path, target_size=(512, 512))
img_array = image.img_to_array(img)

# Create a batch of 1: shape becomes (1, 512, 512, 3)
img_batch = np.expand_dims(img_array, axis=0)

# ⚠️ Apply DenseNet preprocessing so the model reads it correctly
sample_image = preprocess_input(img_batch.copy())

# Generate Heatmap
heatmap = make_gradcam_heatmap(sample_image, grad_model)

# Make a prediction
prediction = model.predict(sample_image, verbose=0)[0][0]
pred_class = 1 if prediction > 0.5 else 0

# Visual Overlay setup
# Since we have the raw img_array, we can just scale it to [0,1] for easy plotting
img_vis = img_array / 255.0

# Resize heatmap to match the original image dimensions dynamically
heatmap_resized = tf.image.resize(heatmap[..., tf.newaxis], (img_vis.shape[0], img_vis.shape[1])).numpy()
heatmap_resized = np.squeeze(heatmap_resized)

# Colorize the heatmap
cmap = cm.get_cmap("jet")
heatmap_colored = cmap(heatmap_resized)[..., :3]

# Superimpose heatmap onto original image (0.6 image, 0.4 heatmap opacity)
overlay = img_vis * 0.6 + heatmap_colored * 0.4
overlay = np.clip(overlay, 0, 1)

# Plotting
class_names = ["Normal", "Pneumonia"] # Ensure class names are defined

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_vis)
axes[0].set_title(f"Original Image\nActual: {actual_label}")
axes[0].axis("off")

axes[1].imshow(heatmap_resized, cmap="jet")
axes[1].set_title("Grad-CAM Heatmap")
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title(f"Overlay\nPredicted: {class_names[pred_class]} (Conf: {prediction:.2f})")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from skimage.segmentation import mark_boundaries
import lime
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.densenet import preprocess_input

# ==========================================
# 🖼️ Load Custom Image & Setup LIME
# ==========================================
print("\n=== 🟢 Generating LIME Explanation ===")

# 👇 CHANGE THIS TO YOUR SPECIFIC IMAGE PATH
custom_image_path = "/content/drive/MyDrive/my_4k_dataset/Pneumonia/CP_1113_3331_0054.png"

# Optional: Set what the actual label is manually so the plot title is correct
actual_label = "Pneumonia"
class_names = ["Normal", "Pneumonia"]

# Load and format the image to 512x512
img = image.load_img(custom_image_path, target_size=(512, 512))
img_array = image.img_to_array(img)

# LIME expects the base image to be in the [0, 1] range for visualization and math
sample_image = img_array / 255.0

# ==========================================
# 🧠 Define LIME Explainer & Wrapper
# ==========================================
explainer = lime_image.LimeImageExplainer()

# Custom SLIC segmenter optimized for medical scans
segmenter = SegmentationAlgorithm('slic', n_segments=100, compactness=10, sigma=1)

def predict_function(images_array):
    # LIME passes arrays in [0, 1]. Scale up to 255, then let DenseNet preprocess it.
    images_255 = images_array * 255.0
    processed_images = preprocess_input(images_255)

    # Predict the perturbed batch
    preds = model.predict(processed_images, batch_size=16, verbose=0)

    # Convert single probability [p] to [[1-p, p]]
    return np.hstack((1 - preds, preds))

# ==========================================
# 🔍 Run LIME & Visualize
# ==========================================
print(f"🔍 Analyzing custom image: {actual_label}... (Generating perturbed samples)")

# Generate the explanation
# ⚠️ Note: num_samples=300 is for speed. Bump to 1000 for your final DS357 report figures!
explanation = explainer.explain_instance(
    sample_image.astype('double'),
    predict_function,
    top_labels=1,
    hide_color=0,
    num_samples=300,
    segmentation_fn=segmenter
)

# Extract the Superpixel Masks
predicted_class = explanation.top_labels[0]

# Get Positive features (Green - regions that pushed the model to this decision)
temp_pos, mask_pos = explanation.get_image_and_mask(
    predicted_class, positive_only=True, num_features=5, hide_rest=False
)

# Get All top features (Positive=Green, Negative=Red)
temp_all, mask_all = explanation.get_image_and_mask(
    predicted_class, positive_only=False, num_features=5, hide_rest=False
)

# Safely clip image values to [0, 1] for matplotlib plotting
safe_temp_pos = np.clip(temp_pos, 0.0, 1.0)
safe_temp_all = np.clip(temp_all, 0.0, 1.0)

# ==========================================
# 📊 Plotting
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Original Image
axes[0].imshow(sample_image)
axes[0].set_title(f"Original Image\nActual: {actual_label}")
axes[0].axis('off')

# Plot 1: Only supportive regions
axes[1].imshow(mark_boundaries(safe_temp_pos, mask_pos))
axes[1].set_title(f"LIME: Supportive Regions Only\nPredicted: {class_names[predicted_class]}")
axes[1].axis('off')

# Plot 2: Supportive and contradicting regions
# Calculate confidence for the title
confidence = predict_function(np.expand_dims(sample_image, axis=0))[0][predicted_class]

axes[2].imshow(mark_boundaries(safe_temp_all, mask_all))
axes[2].set_title(f"LIME: Top 5 Features (Green=Support, Red=Contradict)\nConfidence: {confidence:.2f}")
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.densenet import preprocess_input

# ==========================================
# 🖼️ Load Custom Image
# ==========================================
print("\n=== 🔴 Setup RISE Explainer ===")

# 👇 CHANGE THIS TO YOUR SPECIFIC IMAGE PATH
custom_image_path = "/content/drive/MyDrive/my_4k_dataset/Pneumonia/CP_1113_3331_0054.png"

actual_label = "Pneumonia"
class_names = ["Normal", "Pneumonia"]

# Load and format the image to 512x512
img = image.load_img(custom_image_path, target_size=(512, 512))
img_array = image.img_to_array(img) # Raw [0, 255] array

# Get the base prediction to know which class we are explaining
img_preprocessed = preprocess_input(np.expand_dims(img_array.copy(), axis=0))
base_pred = model.predict(img_preprocessed, verbose=0)[0][0]
pred_class = 1 if base_pred > 0.5 else 0

print(f"✅ Image loaded. Model predicts: {class_names[pred_class]} (Confidence: {base_pred:.4f})")

# ==========================================
# 🧠 Define RISE Algorithm
# ==========================================
def explain_with_rise(model, image_raw, target_class, num_masks=1000, grid_size=16, p_keep=0.5):
    img_h, img_w = image_raw.shape[:2]

    print(f"🔍 Generating {num_masks} masks and processing... (This will take a moment)")

    # 1. Generate small random grids
    small_masks = np.random.rand(num_masks, grid_size, grid_size) < p_keep
    small_masks = small_masks.astype(np.float32)

    # 2. Upsample using Bilinear Interpolation to 512x512
    masks = tf.image.resize(small_masks[..., tf.newaxis], (img_h, img_w), method='bilinear')
    masks = tf.squeeze(masks).numpy()

    # 3. Apply masks to the RAW [0, 255] image
    masked_images = image_raw * masks[..., tf.newaxis]

    # 4. Predict on all masked images in batches to prevent GPU OOM crashes
    preds = []
    batch_size = 16
    for i in range(0, num_masks, batch_size):
        batch = masked_images[i : i + batch_size]

        # Apply DenseNet preprocessing AFTER masking, right before predicting
        batch_preprocessed = preprocess_input(batch.copy())

        batch_preds = model.predict(batch_preprocessed, verbose=0)
        preds.extend(batch_preds)

    preds = np.array(preds).flatten()

    # 5. Flip probabilities if we are explaining the "Normal" (0) class
    if target_class == 0:
        preds = 1.0 - preds

    # 6. Generate the heatmap by calculating the weighted sum of masks
    heatmap = np.zeros((img_h, img_w))
    for i in range(num_masks):
        heatmap += masks[i] * preds[i]

    # 7. Normalize the heatmap to [0, 1] range
    heatmap = heatmap / (num_masks * p_keep)
    heatmap -= heatmap.min()

    max_val = heatmap.max()
    if max_val == 0: max_val = 1e-8
    heatmap /= max_val

    return heatmap

# ==========================================
# 🔍 Run RISE & Visualize
# ==========================================
# ⚠️ Note: num_masks=1000 is optimal, but you can lower it to 300 for faster testing
heatmap = explain_with_rise(model, img_array, target_class=pred_class, num_masks=1000, grid_size=16, p_keep=0.5)

# Visual Overlay setup
img_vis = img_array / 255.0  # Scale raw image to [0,1] for plotting

# Colorize the heatmap
cmap = cm.get_cmap("jet")
heatmap_colored = cmap(heatmap)[..., :3]

# Superimpose heatmap onto original image (0.6 image, 0.4 heatmap opacity)
overlay = np.clip(img_vis * 0.6 + heatmap_colored * 0.4, 0, 1)

# ==========================================
# 📊 Plotting
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_vis)
axes[0].set_title(f"Original Image\nActual: {actual_label}")
axes[0].axis("off")

axes[1].imshow(heatmap, cmap="jet")
axes[1].set_title("RISE Heatmap")
axes[1].axis("off")

axes[2].imshow(overlay)
axes[2].set_title(f"Overlay\nPredicted: {class_names[pred_class]}")
axes[2].axis("off")

plt.tight_layout()
plt.show()